# Chapter 13 — Claude Code Built-in Tools · Homework (self-test)

**Goal:** check that you can choose the right built-in tool with the right input, recover from Edit failures safely, trace callers through wrapper modules, investigate incrementally within a read budget, scope tools with permission rules, and turn tool evidence into modernization priorities.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 75–105 minutes
- Estimated API cost: **$0**. Everything runs offline against the same simulated repository as the practice notebook; no cell calls the API, the Agent SDK or the `claude` CLI.
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**

1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), chosen to mirror the Claude exam's scaled cut score of 720/1,000 (a scaled score, so not a raw-percentage equivalent).

In [ ]:
import hashlib
import json
import os
import re
from pathlib import Path
from typing import Callable

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")   # not used: this homework is fully offline
print("Offline homework | MODEL (for reference) =", MODEL)

### Helpers (run, don't edit)

The practice notebook's `FakeRepo` simulator and `acme-shop` repository, plus a homework-only `app/pricing/` package behind a wrapper `__init__.py`, the simplified permission evaluator (`decide`, `lint_rules`), `hw_repo()` (a fresh repository), `fan_in(repo, module)` and the grading bookkeeping.

In [ ]:
# (offline) Simulated built-in tools
# (offline) A fake repository with simulated Claude Code built-in tools.
# The behavior follows the documented contracts in the tools reference (code.claude.com/docs/en/tools-reference):
# Glob matches paths (sorted by modification time, capped at 100, ignores .gitignore); Grep searches contents
# (regex, three output modes, respects .gitignore unless `path` names the file itself); Read returns numbered lines and a PARTIAL view for big files;
# Write overwrites whole files; Edit needs read-before-edit, an exact match and a unique match (or replace_all).
# Read-before-edit/overwrite is modeled as always required, as on Opus 4.6, Haiku 4.5 and older models (and the exam);
# newer models may skip it when reading would not prompt, but notebooks and PARTIAL views always need the read.
# Error texts are written for this simulator; the real tool's wording differs.
import fnmatch
import re
from dataclasses import dataclass, field

GLOB_CAP = 100
TYPE_EXTS = {"py": (".py",), "ts": (".ts", ".tsx"), "js": (".js", ".jsx"), "md": (".md",),
             "yaml": (".yaml", ".yml"), "json": (".json",), "toml": (".toml",)}


@dataclass
class ToolResult:
    tool: str
    input: dict
    ok: bool
    output: str

    @property
    def tokens(self) -> int:
        """Rough context cost of this result (about 4 characters per token)."""
        return max(1, len(self.output) // 4)

    def __repr__(self) -> str:
        head = self.output if len(self.output) < 160 else self.output[:157] + "..."
        return f"{self.tool}{'' if self.ok else ' [ERROR]'} {self.input} -> {head!r}"


def glob_to_regex(pattern: str) -> re.Pattern:
    """Glob syntax: ** spans directories, * and ? stay inside one segment, {a,b} alternates."""
    i, out = 0, []
    while i < len(pattern):
        c = pattern[i]
        if pattern.startswith("**/", i):
            out.append("(?:.*/)?"); i += 3; continue
        if pattern.startswith("**", i):
            out.append(".*"); i += 2; continue
        if c == "*":
            out.append("[^/]*")
        elif c == "?":
            out.append("[^/]")
        elif c == "{":
            j = pattern.index("}", i)
            out.append("(?:" + "|".join(re.escape(p) for p in pattern[i + 1:j].split(",")) + ")"); i = j
        else:
            out.append(re.escape(c))
        i += 1
    return re.compile("".join(out) + r"\Z")


class FakeRepo:
    """An in-memory repository plus the six core tools. Every call is logged in self.ledger."""

    def __init__(self, files: dict[str, str], gitignore: tuple[str, ...] = (), read_token_limit: int = 2500,
                 git_log: dict[str, int] | None = None):
        self.files = dict(files)
        self.mtime = {p: i for i, p in enumerate(self.files)}
        self.clock = len(self.files)
        self.gitignore = tuple(gitignore)
        self.read_token_limit = read_token_limit
        self.git_log = dict(git_log or {})      # path -> number of commits in the last 12 months
        self.read_at: dict[str, int] = {}       # path -> mtime of the version Claude has fully seen
        self.ledger: list[ToolResult] = []

    # ---------- helpers ----------
    def _tick(self, path: str) -> None:
        self.clock += 1
        self.mtime[path] = self.clock

    def _ignored(self, path: str) -> bool:
        parts = path.split("/")
        for pat in self.gitignore:
            if pat.endswith("/"):
                if pat.rstrip("/") in parts[:-1]:
                    return True
            elif any(fnmatch.fnmatch(seg, pat) for seg in parts):
                return True
        return False

    def _under(self, path: str, base: str | None) -> bool:
        base = (base or "").strip("/")
        return not base or path == base or path.startswith(base + "/")

    def is_dir(self, path: str) -> bool:
        p = path.strip("/")
        return p == "" or any(f.startswith(p + "/") for f in self.files)

    def external_change(self, path: str, content: str) -> None:
        """Simulate a teammate or formatter changing a file on disk (not logged as a tool call)."""
        self.files[path] = content
        self._tick(path)

    def context_tokens(self) -> int:
        return sum(r.tokens for r in self.ledger)

    def calls(self, tool: str | None = None) -> list[ToolResult]:
        return [r for r in self.ledger if tool is None or r.tool == tool]

    def reset_ledger(self) -> None:
        self.ledger.clear()

    # ---------- dispatcher ----------
    def call(self, tool: str, **inp) -> ToolResult:
        handler = {"Glob": self._glob, "Grep": self._grep, "Read": self._read, "Write": self._write,
                   "Edit": self._edit, "Bash": self._bash}.get(tool)
        if handler is None:
            res = ToolResult(tool, inp, False, f"Error: no such tool in this simulator: {tool}")
        else:
            try:
                ok, out = handler(**inp)
            except TypeError as exc:                     # wrong or missing input fields
                ok, out = False, f"InputValidationError: {exc}"
            res = ToolResult(tool, inp, ok, out)
        self.ledger.append(res)
        return res

    def read_raw(self, path: str) -> str | None:
        """Read through the Read tool (logged), then return the raw file text without line numbers."""
        r = self.call("Read", file_path=path)
        if not r.ok or "PARTIAL view" in r.output:
            return None
        return self.files[path]

    # ---------- tools ----------
    def _glob(self, pattern: str, path: str | None = None):
        rx = glob_to_regex(pattern)
        base = (path or "").strip("/")
        hits = [f for f in self.files if self._under(f, base) and rx.match(f[len(base) + 1:] if base else f)]
        hits.sort(key=lambda f: -self.mtime[f])          # newest first in this simulator
        truncated = len(hits) > GLOB_CAP
        lines = hits[:GLOB_CAP]
        if truncated:
            lines.append(f"(Results are truncated: showing {GLOB_CAP} of {len(hits)}. Use a more specific pattern.)")
        return True, "\n".join(lines) if lines else "No files found"

    def _grep(self, pattern: str, path: str | None = None, glob: str | None = None, type: str | None = None,
              output_mode: str = "files_with_matches", head_limit: int | None = None, offset: int = 0,
              multiline: bool = False, **flags):
        if output_mode not in ("files_with_matches", "content", "count"):
            return False, f"Error: invalid output_mode {output_mode!r}"
        re_flags = (re.IGNORECASE if flags.get("-i") else 0) | (re.MULTILINE | re.DOTALL if multiline else 0)
        try:
            rx = re.compile(pattern, re_flags)
        except re.error as exc:
            return False, f"Error: regex parse error in {pattern!r}: {exc}"
        if glob:
            g = glob_to_regex(glob if "/" in glob else "**/" + glob)
        exts = TYPE_EXTS.get(type, ()) if type else ()
        if type and not exts:
            return False, f"Error: unrecognized file type: {type}"
        explicit = (path or "").strip("/")                 # a path naming one file is searched even if gitignored
        files = [f for f in sorted(self.files) if self._under(f, path) and (f == explicit or not self._ignored(f))
                 and (not glob or g.match(f)) and (not exts or f.endswith(exts))]
        before, after = flags.get("-B", flags.get("-C", 0)) or 0, flags.get("-A", flags.get("-C", 0)) or 0
        entries, counts = [], {}
        for f in files:
            text = self.files[f]
            if multiline:
                n = len(rx.findall(text))
                if n:
                    counts[f] = n
                    if output_mode == "content":
                        for m in rx.finditer(text):
                            line_no = text.count("\n", 0, m.start()) + 1
                            entries.append(f"{f}:{line_no}:{m.group(0)}")
                continue
            lines = text.splitlines()
            hit_lines = [i for i, ln in enumerate(lines) if rx.search(ln)]
            if not hit_lines:
                continue
            counts[f] = len(hit_lines)
            if output_mode == "content":
                shown = sorted({j for i in hit_lines for j in range(max(0, i - before), min(len(lines), i + after + 1))})
                entries += [f"{f}:{j + 1}:{lines[j]}" if j in hit_lines else f"{f}-{j + 1}-{lines[j]}" for j in shown]
        if output_mode == "files_with_matches":
            entries = list(counts)
        elif output_mode == "count":
            entries = [f"{f}:{n}" for f, n in counts.items()]
        total = len(entries)
        page = entries[offset: offset + head_limit if head_limit else None]
        if not page:
            return True, "No matches found" if total == 0 else "No entries at this offset"
        out = "\n".join(page)
        if output_mode == "count":
            out += f"\nTotal: {sum(counts.values())} matches in {len(counts)} files"
        elif len(page) < total:
            out += f"\n(showing {len(page)} of {total}; use offset to see more)"
        return True, out

    def _read(self, file_path: str, offset: int | None = None, limit: int | None = None):
        if self.is_dir(file_path) and file_path not in self.files:
            return False, f"Error: {file_path} is a directory. Read only reads files; list directories with ls via Bash."
        if file_path not in self.files:
            return False, f"Error: file does not exist: {file_path}"
        lines = self.files[file_path].splitlines()
        if not lines:
            self.read_at[file_path] = self.mtime[file_path]
            return True, "(the file exists but its contents are empty)"
        start = (offset or 1) - 1
        if start >= len(lines):
            return True, f"(offset is past the end: the file has {len(lines)} lines)"
        end = len(lines) if limit is None else min(len(lines), start + limit)
        body = "\n".join(f"{i + 1:>6}\t{lines[i]}" for i in range(start, end))
        if offset is None and limit is None and len(body) // 4 > self.read_token_limit:
            cut, size = [], 0
            for i, ln in enumerate(lines):
                row = f"{i + 1:>6}\t{ln}"
                if (size + len(row)) // 4 > self.read_token_limit:
                    break
                cut.append(row); size += len(row) + 1
            return True, "\n".join(cut) + (f"\n[PARTIAL view: showing lines 1-{len(cut)} of {len(lines)}. "
                                           f"Read more with offset and limit, or Grep for what you need.]")
        self.read_at[file_path] = self.mtime[file_path]
        return True, body

    def _fresh(self, path: str) -> bool:
        return self.read_at.get(path) == self.mtime.get(path)

    def _write(self, file_path: str, content: str):
        if file_path in self.files and file_path not in self.read_at:
            return False, "Error: file has not been read yet. Read it before overwriting it."
        existed = file_path in self.files
        self.files[file_path] = content
        self._tick(file_path)
        self.read_at[file_path] = self.mtime[file_path]
        return True, f"{'Overwrote' if existed else 'Created'} {file_path} ({len(content.splitlines())} lines)"

    def _edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False):
        if file_path not in self.files:
            return False, f"Error: file does not exist: {file_path}"
        if file_path not in self.read_at:
            return False, "Error: file has not been read yet. Read it first before editing it."
        if old_string == new_string:
            return False, "Error: old_string and new_string are identical; nothing to change."
        text = self.files[file_path]
        n = text.count(old_string)
        stale = not self._fresh(file_path)
        if n == 0:
            hint = " The file changed after you read it; read it again." if stale else \
                " Check whitespace and indentation; copy the text exactly from a fresh Read."
            return False, "Error: string to replace not found in file." + hint
        if n > 1 and not replace_all:
            return False, (f"Error: found {n} matches of the string to replace, but replace_all is false. "
                           "To replace all occurrences, set replace_all to true. To replace only one, "
                           "provide more surrounding context so old_string matches exactly once.")
        self.files[file_path] = text.replace(old_string, new_string) if replace_all else text.replace(old_string, new_string, 1)
        self._tick(file_path)
        self.read_at[file_path] = self.mtime[file_path]
        note = " Note: the file had changed since your last read; re-read it before edits that depend on context." if stale else ""
        return True, f"Edited {file_path}: replaced {n if replace_all else 1} occurrence(s).{note}"

    def _bash(self, command: str, timeout: int | None = None, description: str | None = None,
              run_in_background: bool = False):
        cmd = command.strip()
        if cmd == "ls" or cmd.startswith("ls "):
            base = cmd[2:].strip().strip("/")
            if base and not self.is_dir(base):
                return False, f"ls: {base}: No such file or directory"
            names = sorted({f[len(base) + 1 if base else 0:].split("/")[0] + ("/" if "/" in f[len(base) + 1 if base else 0:] else "")
                            for f in self.files if self._under(f, base)})
            return True, "\n".join(names)
        m = re.fullmatch(r"(cat|head|tail) (\S+)", cmd)
        if m:   # a plain single-file view also satisfies read-before-edit
            path = m.group(2)
            if path not in self.files:
                return False, f"{m.group(1)}: {path}: No such file or directory"
            self.read_at[path] = self.mtime[path]
            lines = self.files[path].splitlines()
            lines = lines[:10] if m.group(1) == "head" else lines[-10:] if m.group(1) == "tail" else lines
            return True, "\n".join(lines)
        m = re.fullmatch(r"wc -l (\S+)", cmd)
        if m and m.group(1) in self.files:
            return True, f"{len(self.files[m.group(1)].splitlines())} {m.group(1)}"
        if cmd.startswith("git log") and "--name-only" in cmd:
            return True, "\n".join(f"{n:>4} commits  {p}" for p, n in sorted(self.git_log.items(), key=lambda kv: -kv[1]))
        if cmd.startswith(("pytest", "python -m pytest")):
            tests = [f for f in self.files if re.search(r"(^|/)test_[^/]*\.py$", f)]
            return True, f"{'.' * len(tests)}\n{len(tests)} passed in 0.42s (simulated)"
        if cmd.startswith(("git diff", "git status")):
            return True, "(simulated: nothing to show)"
        return False, f"(simulated shell) command not supported here: {cmd}"


# (offline) The acme-shop repository
# (offline) acme-shop: a small "legacy" repository used by every exercise in this chapter.
ACME_FILES: dict[str, str] = {
    "README.md": "# acme-shop\n\nOrders, billing and refunds for the Acme web shop. Python backend in app/, React front end in web/.\n",
    "pyproject.toml": "[project]\nname = \"acme-shop\"\nversion = \"3.4.1\"\ndependencies = [\"flask>=3\", \"requests\"]\n",
    ".gitignore": "build/\n*.log\nnode_modules/\n",
    "app/__init__.py": "",
    "app/api/__init__.py": "",
    "app/api/routes.py": '''from app.billing import issue_refund
from app.orders.service import get_order
from app.web import route


@route("/orders/<order_id>/refund", methods=["POST"])
def refund_order(order_id: str, amount: float):
    order = get_order(order_id)
    return issue_refund(order.id, amount)


@route("/orders/<order_id>", methods=["GET"])
def show_order(order_id: str):
    return get_order(order_id)
''',
    "app/web.py": '''def route(path, methods=("GET",)):
    def wrap(fn):
        fn.route = (path, methods)
        return fn
    return wrap
''',
    "app/orders/__init__.py": "",
    "app/orders/service.py": '''from app.orders.models import Order


def get_order(order_id: str) -> Order:
    return Order.load(order_id)


def cancel_order(order_id: str) -> None:
    Order.load(order_id).cancel()
''',
    "app/orders/models.py": '''class Order:
    def __init__(self, id: str, total: float):
        self.id, self.total = id, total

    @classmethod
    def load(cls, order_id: str) -> "Order":
        return cls(order_id, 120.0)

    def cancel(self) -> None:
        self.status = "cancelled"
''',
    "app/billing/__init__.py": '''"""Public billing API. Import from here, not from the submodules."""
from .refunds import create_refund as issue_refund, cancel_refund
from .ledger import post_entry

__all__ = ["issue_refund", "cancel_refund", "post_entry"]
''',
    "app/billing/refunds.py": '''from app.billing.ledger import post_entry
from app.billing.policy import check_policy
from app.payments.gateway import Gateway


def create_refund(order_id: str, amount: float) -> dict:
    check_policy(order_id, amount)
    receipt = Gateway().submit_refund(order_id, amount)
    post_entry(order_id, -amount, memo="refund")
    return {"order_id": order_id, "amount": amount, "receipt": receipt}


def cancel_refund(refund_id: str) -> None:
    post_entry(refund_id, 0.0, memo="refund cancelled")
''',
    "app/billing/ledger.py": '''ACCOUNTS = {"closed": set()}


def post_entry(account: str, amount: float, memo: str = "") -> None:
    if account in ACCOUNTS["closed"]:
        raise ValueError("Ledger entry rejected: account closed")
    print(f"ledger {account} {amount:+.2f} {memo}")
''',
    "app/billing/policy.py": '''MAX_REFUND = 500.0


def check_policy(order_id: str, amount: float) -> None:
    if amount > MAX_REFUND:
        raise PermissionError(f"Refund of {amount} exceeds the {MAX_REFUND} limit; escalate to a supervisor")
''',
    "app/payments/__init__.py": "",
    "app/payments/gateway.py": '''import requests


class GatewayError(Exception):
    pass


class Gateway:
    BASE_URL = "https://payments.example.com/v2"

    def charge(self, order_id: str, amount: float) -> str:
        r = requests.post(f"{self.BASE_URL}/charges", json={"order": order_id, "amount": amount}, timeout=30)
        return r.json()["id"]

    def submit_refund(self, order_id: str, amount: float) -> str:
        r = requests.post(f"{self.BASE_URL}/refunds", json={"order": order_id, "amount": amount}, timeout=30)
        if r.status_code == 402:
            raise GatewayError("Refund declined by processor")
        return r.json()["id"]
''',
    "app/support/__init__.py": "",
    "app/support/macros.py": '''import app.billing as billing


def goodwill_refund(order_id: str) -> dict:
    """Support macro: refund 10 per cent of an order as a goodwill gesture."""
    return billing.issue_refund(order_id, 12.0)
''',
    "app/workers/__init__.py": "",
    "app/workers/nightly.py": '''from app.billing import cancel_refund, issue_refund as refund
from app.billing import post_entry


def retry_failed_refunds(failed: list[tuple[str, float]]) -> None:
    for order_id, amount in failed:
        refund(order_id, amount)


def close_day(totals: dict[str, float]) -> None:
    for account, amount in totals.items():
        post_entry(account, amount, memo="end of day")


def expire_pending(refund_ids: list[str]) -> None:
    for rid in refund_ids:
        cancel_refund(rid)
''',
    "app/config/settings.py": '''SERVICES = {
    "payments": {
        "url": "https://payments.example.com/v2",
        "timeout": 30,
        "retries": 3,
    },
    "ledger": {
        "url": "https://ledger.internal/v1",
        "timeout": 30,
        "retries": 3,
    },
    "search": {
        "url": "https://search.internal/v1",
        "timeout": 10,
        "retries": 1,
    },
}
''',
    "deploy/services.yaml": '''services:
  - name: api
    resources:
      replicas: 2
      maxSurge: 1
  - name: billing
    resources:
      replicas: 2
      maxSurge: 1
  - name: worker
    resources:
      replicas: 2
      maxSurge: 1
''',
    "deploy/nginx.conf": """server {
    listen 80;
    server_name shop.example.com;

    location /api/ {
        proxy_pass http://api:8000;
        proxy_read_timeout 30s;
    }
}

server {
    listen 443 ssl;
    server_name shop.example.com;

    location /api/ {
        proxy_pass http://api:8000;
        proxy_read_timeout 30s;
    }
}
""",
    "legacy/old_refunds.py": '''# Deprecated in 2019; kept for reference. Not imported anywhere.
def create_refund(order_id, amount):
    raise NotImplementedError("use app.billing")
''',
    "tests/test_refunds.py": '''from app.billing.refunds import create_refund


def test_small_refund_ok():
    assert create_refund("A-1", 10.0)["amount"] == 10.0
''',
    "tests/test_ledger.py": '''from app.billing.ledger import post_entry


def test_post_entry_prints(capsys):
    post_entry("A-1", 5.0)
    assert "A-1" in capsys.readouterr().out
''',
    "build/lib/app/payments/gateway.py": '# build artefact (gitignored)\nraise GatewayError("Refund declined by processor")\n',
    "web/package.json": '{"name": "acme-web", "scripts": {"test": "vitest", "lint": "eslint src"}}\n',
    "web/src/components/RefundButton.tsx": "export function RefundButton() { return <button>Refund</button>; }\n",
    "web/src/components/RefundButton.test.tsx": "import { RefundButton } from './RefundButton';\ntest('renders', () => {});\n",
    "web/src/components/OrderList.tsx": "export function OrderList() { return <ul />; }\n",
    "web/src/components/OrderList.test.tsx": "import { OrderList } from './OrderList';\ntest('renders', () => {});\n",
    "web/src/pages/orders/OrderPage.tsx": "import { OrderList } from '../../components/OrderList';\nexport default function OrderPage() { return <OrderList />; }\n",
    "web/src/pages/orders/OrderPage.test.tsx": "import OrderPage from './OrderPage';\ntest('renders', () => {});\n",
    "notebooks/refund_analysis.ipynb": '{"cells": [], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}\n',
}

# 110 generated icon components: enough to hit the 100-result Glob cap
for _i in range(110):
    ACME_FILES[f"web/src/icons/Icon{_i:03d}.tsx"] = f"export const Icon{_i:03d} = () => <svg data-id='{_i}' />;\n"

# One big generated file: enough to trigger Read's PARTIAL view
ACME_FILES["app/generated/rates_table.py"] = "RATES = {\n" + "".join(
    f"    \"{c}{n:03d}\": {1 + n / 1000:.3f},\n" for c in "ABCDEFGH" for n in range(160)) + "}\n"

ACME_GITIGNORE = ("build/", "*.log", "node_modules/")

# Commits per file in the last 12 months (what `git log --name-only` would tell you)
ACME_GIT_LOG = {
    "app/billing/refunds.py": 41, "app/payments/gateway.py": 23, "app/api/routes.py": 17,
    "app/billing/ledger.py": 9, "app/orders/service.py": 6, "app/billing/policy.py": 12,
    "app/workers/nightly.py": 14, "app/support/macros.py": 3, "app/orders/models.py": 2,
}


# (offline) Homework additions: a pricing package behind a wrapper module
# (offline) Extra files for the homework: a pricing package with a wrapper __init__ and a few consumers
HW_EXTRA_FILES: dict[str, str] = {
    "app/pricing/__init__.py": '''"""Pricing facade: import from app.pricing, never from the submodules."""
from .engine import (
    compute_price as quote,
    apply_discount,
)
from .tax import vat_for as tax_rate

PRICE_VERSION = "2026.10"

__all__ = ["quote", "apply_discount", "tax_rate", "round_money", "PRICE_VERSION"]

from .money import round_money
''',
    "app/pricing/engine.py": '''from app.pricing.money import round_money


class CouponError(Exception):
    pass


def compute_price(sku: str, qty: int) -> float:
    return round_money(9.99 * qty)


def apply_discount(price: float, coupon: str) -> float:
    if coupon.startswith("OLD-"):
        raise CouponError("Coupon expired (code reuse)")
    return round_money(price * 0.9)


def bulk_price(sku: str, qty: int, coupon: str) -> float:
    return apply_discount(compute_price(sku, qty), coupon)
''',
    "app/pricing/tax.py": "def vat_for(country: str) -> float:\n    return {\"DE\": 0.19, \"FR\": 0.20}.get(country, 0.0)\n",
    "app/pricing/money.py": "def round_money(x: float) -> float:\n    return round(x + 1e-9, 2)\n",
    "app/api/pricing_routes.py": '''from app.pricing import PRICE_VERSION, quote, tax_rate
from app.web import route


@route("/quote/<sku>", methods=["GET"])
def get_quote(sku: str, qty: int = 1, country: str = "DE"):
    net = quote(sku, qty)
    return {"net": net, "vat": net * tax_rate(country), "version": PRICE_VERSION}
''',
    "app/checkout/__init__.py": "",
    "app/checkout/cart.py": '''import app.pricing as pricing


def checkout_total(prices: list[float], coupon: str | None) -> float:
    total = sum(prices)
    if coupon:
        total = pricing.apply_discount(total, coupon)
    return pricing.round_money(total)
''',
    "app/reports/__init__.py": "",
    "app/reports/monthly.py": '''from app.pricing import quote as price_of


def catalogue_value(skus: list[str]) -> float:
    return sum(price_of(s, 1) for s in skus)
''',
    "tests/test_pricing.py": '''from app.pricing import apply_discount


def test_discount():
    assert apply_discount(100.0, "NEW-1") == 90.0


def test_expired():
    try:
        apply_discount(100.0, "OLD-1")
    except Exception:
        pass
''',
}


# (offline) Simplified permission-rule evaluator (see the practice notebook, 13.10)
# (offline) A SIMPLIFIED model of Claude Code permission rules, for learning only.
# Modeled on code.claude.com/docs/en/permissions: deny -> ask -> allow, first match wins; bare tool names match
# every use; Bash rules match command text with * wildcards; a compound command is split into subcommands
# and EACH subcommand is decided against the whole rule set (any deny -> deny, any ask -> ask, allow only when
# every subcommand is allowed by some allow rule or is read-only); a command ending in && or || is not split,
# so allow rules do not approve it; Read/Edit DENY rules also apply to the files named by Bash file commands
# that Claude Code recognizes (cat/head/tail/sed read, sed -i and tee write) and to redirection targets
# (< file reads, > file writes), but not to arbitrary subprocesses (python script.py) or unnamed files
# (grep -r . from the parent directory); Read(...) rules cover Read/Grep/Glob/LSP, Edit(...) rules cover Edit/Write/NotebookEdit; a Read deny also
# blocks Edit and Write (not NotebookEdit); an Edit allow also grants Read; Write(...), Glob(...) and Grep(...) path rules
# are never consulted; WebFetch(domain:...) matches the host; mcp__server and tool-name globs match MCP tools.
# Not modeled: modes, hooks, sandbox, wrappers, subshells, symlinks, additional directories, settings
# precedence; the recognized Bash file commands are a small assumed subset of the real list. Use the real `/permissions` UI to confirm behavior.
# Assumptions where the docs are silent (checked 2026-10-03): the docs say only "read-only forms of git", so
# the git subset below is a guess; `Read(./.env)` is documented as "the .env file in the current directory"
# and is modeled as anchored there (no doc example covers nested files, so write `Read(**/.env)` when you mean
# every .env); a bare `Edit` (or `Write`) deny is modeled as matching only that one tool.
import fnmatch
import re
import shlex
from urllib.parse import urlparse

READ_TOOLS = {"Read", "Grep", "Glob", "LSP"}
READ_DENY_ALSO_BLOCKS = {"Edit", "Write"}          # the docs: a Read deny does not cover NotebookEdit
EDIT_TOOLS = {"Edit", "Write", "NotebookEdit"}
READ_ONLY_BASH = ("ls", "cat", "head", "tail", "grep", "find", "wc", "pwd", "echo", "which", "diff", "stat", "du",
                  "git status", "git log", "git diff", "git show")   # documented examples + an assumed git subset
PATH_KEYS = ("file_path", "notebook_path", "path")
SEPARATORS = r"&&|\|\||\|&|;|\||&|\n"           # compound-command separators named in the docs
BASH_READ_CMDS = ("cat", "head", "tail", "sed")      # recognized file commands (doc examples; an assumed subset)
BASH_WRITE_CMDS = ("tee",)


def parse_rule(rule: str) -> tuple[str, str | None]:
    m = re.fullmatch(r"\s*([A-Za-z_*][\w*]*)\s*(?:\((.*)\))?\s*", rule)
    if not m:
        raise ValueError(f"not a permission rule: {rule!r}")
    return m.group(1), m.group(2)


def _bash_match(spec: str, command: str) -> bool:
    spec = spec[:-2] + " *" if spec.endswith(":*") else spec
    rx = "".join(".*" if ch == "*" else re.escape(ch) for ch in spec)
    if spec.endswith(" *") and spec.count("*") == 1 and command == spec[:-2]:
        return True                                   # "npm run *" also matches bare "npm run"
    return re.fullmatch(rx, command) is not None


def _glob_rx(spec: str) -> str:
    i, out = 0, []
    while i < len(spec):
        if spec.startswith("**/", i):
            out.append("(?:.*/)?"); i += 3; continue
        if spec.startswith("/**", i) and i + 3 == len(spec):
            out.append("(?:/.*)?"); i += 3; continue
        if spec.startswith("**", i):
            out.append(".*"); i += 2; continue
        c = spec[i]
        out.append("[^/]*" if c == "*" else "[^/]" if c == "?" else re.escape(c)); i += 1
    return "".join(out)


def _path_match(spec: str, path: str, kind: str) -> bool:
    """gitignore-flavoured, with the project root as the working directory:
    '//x' absolute, '/x' and './x' anchored at the root, a bare name (no '/') at any depth,
    and a single-directory pattern such as 'secrets/**' at any depth for deny/ask rules only."""
    path = path[2:] if path.startswith("./") else path.lstrip("/")
    anchored = spec.startswith(("/", "./"))
    spec = spec[2:] if spec.startswith("//") else spec[2:] if spec.startswith("./") else spec.lstrip("/")
    body = spec[:-3] if spec.endswith("/**") else spec
    any_depth = not anchored and ("/" not in spec or (kind != "allow" and "/" not in body))
    return re.fullmatch(("(?:.*/)?" if any_depth else "") + _glob_rx(spec), path) is not None


def rule_matches(rule: str, tool: str, inp: dict, kind: str) -> bool:
    name, spec = parse_rule(rule)
    if name in ("Write", "Glob", "Grep", "NotebookEdit", "MultiEdit") and spec:   # never consulted for paths
        return False
    if spec in (None, "", "*"):
        if name == tool or (name.startswith("mcp__") and "*" not in name and tool.startswith(name + "__")):
            return True                                   # a bare tool name, or every tool of one MCP server
        if "*" in name:                                   # tool-name globs: "*", "mcp__*" (deny/ask only);
            anchored = re.match(r"mcp__[^*]+?__", name)   # allow globs need a literal mcp__<server>__ prefix
            return (kind != "allow" or anchored is not None) and fnmatch.fnmatchcase(tool, name)
        return False
    if name == "Bash" and tool in ("Bash", "Monitor"):
        return _bash_match(spec, inp.get("command", "").strip())      # one subcommand (decide() splits)
    if name == "WebFetch" and tool == "WebFetch" and spec.startswith("domain:"):
        host, pat = (urlparse(inp.get("url", "")).hostname or "").lower(), spec[len("domain:"):].lower()   # case-insensitive
        if pat == "*":
            return True
        if pat.startswith("*."):
            return host.endswith(pat[1:])
        return host == pat
    if name == "Agent" and tool == "Agent":
        return inp.get("subagent_type") == spec
    path = next((inp[k] for k in PATH_KEYS if inp.get(k)), None)
    if path is None:
        return False
    if name == "Read" and (tool in READ_TOOLS or (kind == "deny" and tool in READ_DENY_ALSO_BLOCKS)):
        return _path_match(spec, path, kind)
    if name == "Edit" and (tool in EDIT_TOOLS or (kind == "allow" and tool in READ_TOOLS)):
        return _path_match(spec, path, kind)
    return False


def bash_parts(command: str) -> list[str]:
    return [p.strip() for p in re.split(SEPARATORS, command) if p.strip()]


def bash_file_targets(part: str) -> list[tuple[str, str]]:
    """(tool, path) pairs for the files one subcommand names: ('Read', p) for reads, ('Edit', p) for writes."""
    out = []
    for op, target in re.findall(r"(?:^|\s)\d?(>>|>|<)\s*([^\s&|;<>]+)", part):    # redirection targets
        if not target.startswith("&") and target != "/dev/null":
            out.append(("Read" if op == "<" else "Edit", target))
    words = re.sub(r"(?:^|\s)\d?(>>|>|<)\s*[^\s&|;<>]+", " ", part)
    try:
        tokens = shlex.split(words)
    except ValueError:
        tokens = words.split()
    if not tokens:
        return out
    cmd, args = tokens[0], tokens[1:]
    operands = [t for t in args if not t.startswith("-") and not t.isdigit()]
    if cmd == "sed":
        in_place = any(t == "--in-place" or re.match(r"-[a-zA-Z]*i", t) for t in args)
        if not any(t in ("-e", "--expression") or t.startswith("-e") for t in args) and operands:
            operands = operands[1:]                                    # the first operand is the sed script
        out += [("Read", f) for f in operands] + ([("Edit", f) for f in operands] if in_place else [])
    elif cmd in BASH_READ_CMDS:
        out += [("Read", f) for f in operands]
    elif cmd in BASH_WRITE_CMDS:
        out += [("Edit", f) for f in operands]
    return out


def _decide_bash_part(rules: dict, tool: str, part: str, allow_ok: bool) -> str:
    inp = {"command": part}
    if any(rule_matches(r, tool, inp, "deny") for r in rules.get("deny", [])):
        return "deny"
    if tool == "Bash":                                  # Read/Edit deny rules also cover recognized file commands
        for file_tool, path in bash_file_targets(part):
            if any(parse_rule(r)[0] in ("Read", "Edit") and rule_matches(r, file_tool, {"file_path": path}, "deny")
                   for r in rules.get("deny", [])):
                return "deny"
    if any(rule_matches(r, tool, inp, "ask") for r in rules.get("ask", [])):
        return "ask"
    if allow_ok and any(rule_matches(r, tool, inp, "allow") for r in rules.get("allow", [])):
        return "allow"
    if tool == "Bash" and any(part == c or part.startswith(c + " ") for c in READ_ONLY_BASH):
        return "allow"                                  # built-in read-only command
    return "ask"


def default_decision(tool: str, inp: dict) -> str:
    if tool in READ_TOOLS | {"Agent"}:
        return "allow"                                  # inside the working directory
    return "ask"                                        # edits, writes, web fetch/search, other tools


def decide(rules: dict, tool: str, inp: dict) -> str:
    """Return 'deny', 'ask' or 'allow' for one tool call under {'allow': [...], 'ask': [...], 'deny': [...]}."""
    if tool in ("Bash", "Monitor"):
        command = inp.get("command", "")
        parts = bash_parts(command)
        if not parts:
            return "ask"
        allow_ok = re.search(r"(&&|\|\|)\s*$", command) is None      # "npm test &&" is unparseable for allow rules
        decisions = [_decide_bash_part(rules, tool, p, allow_ok) for p in parts]
        return "deny" if "deny" in decisions else "ask" if "ask" in decisions else "allow"
    for kind in ("deny", "ask", "allow"):
        for rule in rules.get(kind, []):
            if rule_matches(rule, tool, inp, kind):
                return kind
    return default_decision(tool, inp)


def lint_rules(rules: dict) -> list[str]:
    """Warnings Claude Code would also give (or that this course recommends)."""
    warnings = []
    for kind, items in rules.items():
        for r in items:
            name, spec = parse_rule(r)
            if name in ("Write", "Glob", "Grep", "NotebookEdit", "MultiEdit") and spec:
                warnings.append(f"{kind}: {r} is never consulted for paths; use "
                                f"{'Read' if name in ('Glob', 'Grep') else 'Edit'}({spec})")
            if kind == "allow" and name == "Bash" and spec and re.match(r"^\S+ \*", spec) and spec.split()[0] in ("git", "npm", "docker", "kubectl"):
                warnings.append(f"allow: {r} puts * right after the program name and allows every subcommand")
    return warnings


# (offline) Repository factories and helpers used by the exercises
def hw_repo() -> FakeRepo:
    return FakeRepo({**ACME_FILES, **HW_EXTRA_FILES}, ACME_GITIGNORE, git_log=ACME_GIT_LOG)


def fan_in(repo: FakeRepo, module: str) -> int:
    """How many other .py files import `module` (absolute dotted import or relative 'from .stem')."""
    stem, dotted = Path(module).stem, module[:-3].replace("/", ".")
    res = repo.call("Grep", pattern=rf"(from|import)\s+({re.escape(dotted)}\b|\.{stem}\b)", type="py")
    return len([f for f in res.output.splitlines() if f.endswith(".py") and f != module])


# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


print("helpers loaded:", len(ACME_FILES) + len(HW_EXTRA_FILES), "files in hw_repo()")

## Part A — Quiz (10 questions)

Pick one letter per question. Domains: built-in tool selection and Edit recovery (Domain 2, task 2.5), built-in vs MCP tools (2.4), large-codebase exploration (Domain 5, task 5.4), modernization planning (1.6), and current permission and Agent SDK behavior.

**Q1. (Developer productivity)** You must update every test that imports the deprecated helper `renderLegacy`. Tests live next to their components in about 40 folders and are named `*.spec.ts`; most of them do not use the helper. Which first step gives exactly the files to change with the least context?

- A) Glob `**/*.spec.ts`, then Read every result to see which ones import the helper.
- B) Grep for `renderLegacy` with `glob: "*.spec.ts"` and `output_mode: "files_with_matches"`.
- C) Grep for `spec` across the whole repository.
- D) Read the test folder's index file and follow its imports.

**Q2. (Developer productivity)** In a large monorepo, an agent runs Glob `**/*.ts`, gets 100 paths plus a truncation notice, sees none under `packages/billing/`, and concludes that the billing package contains no TypeScript. What should it do?

- A) Accept the conclusion; Glob returns every match.
- B) Read `packages/billing` with the Read tool to list its contents.
- C) Grep for the string `.ts` across the repository.
- D) Run a narrower Glob, for example `packages/billing/**/*.ts` (or pass `path: "packages/billing"`), because the first result was capped.

**Q3. (Developer productivity)** A 600-line generated API client contains 14 identical method stubs, each with the line `retries=3,`. Only the `create_invoice` stub must change to `retries=5,`, and each stub's method name is more than 30 lines above that line. Edit with `old_string: "retries=3,"` fails because the text is not unique. What is the most reliable next step?

- A) Set `replace_all: true` so the stubs stay consistent with each other.
- B) Retry the same Edit with an extra space after the comma.
- C) Read the full file, change only the `create_invoice` stub, Write it back and diff.
- D) Run `sed -i` on the `create_invoice` line number from a Read made an hour ago.

**Q4. (Developer productivity)** An Edit fails with "string to replace not found", although the developer can see the text in the file. The agent built `old_string` by copying a line from the Read output, `    42\t    if amount > MAX_REFUND:`. What is the fix?

- A) Copy the text without the line-number prefix, keeping the file's exact indentation, and retry the Edit.
- B) Use `replace_all: true`.
- C) Write the whole file again from memory.
- D) Switch to Glob to locate the line.

**Q5. (Developer productivity)** Asked "how are webhook signatures verified?" in a 1,500-file repository, an agent starts reading every file under `src/`. After about 200 files its answers mention "typical HMAC patterns" instead of the project's own classes. What is the best change to the approach?

- A) Switch to a model with a larger context window and keep reading.
- B) Grep for the webhook route and `signature`, Read only those hits, and follow their imports.
- C) Summarize each file into one paragraph first, then answer from the summaries.
- D) Run `/compact` after every 50 files.

**Q6. (Developer productivity)** Before changing the signature of `chargeCard`, you must find every consumer. The barrel file `packages/payments/index.ts` contains `export { chargeCard as charge } from "./card";`. A Grep for `chargeCard(` finds only the definition and one unit test. What should you do?

- A) Change the signature: the search shows one caller.
- B) Grep for `charge` case-insensitively and update every hit.
- C) Grep for imports and calls of each exported name (here `charge`), following further aliases.
- D) Read every file in the repository that imports anything from `packages/payments`.

**Q7. (Developer productivity)** A team connected an MCP tool `search_runbooks` (description: "Search runbooks.") backed by the live incident wiki. During incidents the agent keeps running Grep over an old `docs/runbooks/` folder in the repository and quotes outdated steps. What is the best fix?

- A) Add a line to the system prompt: "never use Grep".
- B) Deny the Grep tool for this agent.
- C) Add a classifier that routes runbook and incident questions to the MCP tool.
- D) Rewrite the MCP description: live, current wiki data; prefer it over Grep for incidents.

**Q8. (Claude Code configuration, current docs)** You want Claude Code to never read the project's root `.env` file and also never edit or overwrite it with the Edit or Write tools. Which entry in the project's `.claude/settings.json` does that with one rule?

- A) `"deny": ["Read(/.env)"]`, because a Read deny rule also blocks Edit and Write on the same path.
- B) `"deny": ["Write(/.env)"]`.
- C) `"allow": ["Read(!.env)"]`.
- D) A CLAUDE.md line: "Never open or change .env".

**Q9. (Developer productivity)** An agent fixes a bug with `sed -i '212s/limit=10/limit=50/' app/api.py`, using a line number from a Read it made earlier in the session. Meanwhile a formatter rewrote the file. What is the risk, and what is the better tool?

- A) No risk; `sed` and Edit are equivalent.
- B) Line 212 may now hold different code, so the change can land in the wrong place silently; Edit matches the exact text and fails safely instead when the text has changed.
- C) The risk is only performance; use Write so the whole file is replaced.
- D) Use Grep with `multiline: true` to make the replacement.

**Q10. (Developer productivity, modernization)** A customer wants to move a 700-file monolith from an old web framework version to the current one. Where should the engagement start?

- A) Ask Claude what changed between the versions and apply every change in one pass.
- B) Upgrade the files in alphabetical order so nothing is missed.
- C) Count framework imports and deprecated APIs per file, add churn and test gaps, then plan.
- D) Read all 700 files in one session so Claude has full context, then start editing.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'd14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b', 'q3': '06217269d5c7af55ec1458d68670a9d1686eee2fa64726f5ac13566f0efad593', 'q4': '0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0', 'q5': 'fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c', 'q6': '18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc', 'q9': 'b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e', 'q10': 'd6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker. Use the tools through `repo.call("Grep", ...)`, `repo.call("Read", ...)`, `repo.read_raw(path)` (a logged Read that returns the raw text), `repo.call("Write", ...)` and `repo.call("Edit", ...)`: several checkers inspect `repo.ledger` to see *how* you worked, not only what you returned.

### Exercise 1 — The right tool with the right input

Write `select_tool(task)` returning `(tool_name, tool_input)` with the **real** input field names:

| `task["kind"]` | Other keys | Expected call |
|---|---|---|
| `find_files` | `pattern`, optional `under` | `Glob` with `pattern` (+ `path`) |
| `find_text` | `text`, optional `file_type` | `Grep`: the text **escaped** as a regex, `output_mode: "content"` (+ `type`) |
| `files_containing` | `text` | `Grep`, escaped pattern, paths only (`files_with_matches`) |
| `count_matches` | `text`, `under` | `Grep`, escaped pattern, `path`, `output_mode: "count"` |
| `read_file` | `path` | `Read` with `file_path` |
| `create_file` | `path`, `content` | `Write` with `file_path`, `content` |
| `replace_once` / `replace_every` | `path`, `old`, `new` | `Edit` with `file_path`, `old_string`, `new_string` (+ `replace_all: true` for every occurrence) |
| `run` | `command` | `Bash` with `command` |
| `edit_notebook_cell` | `path`, `cell_id`, `source` | `NotebookEdit` with `notebook_path`, `cell_id`, `new_source` |
| `fetch_page` | `url`, `question` | `WebFetch` with `url`, `prompt` |
| `web_search` | `query` | `WebSearch` with `query` |
| `delegate_exploration` | `question` | `Agent` with `description`, `prompt`, `subagent_type: "Explore"` |

Theory: [13.1](README.md#131-the-six-core-tools-and-how-claude-chooses-among-them), [13.2](README.md#132-grep-vs-glob-content-search-vs-path-search), [13.9](README.md#139-other-current-built-in-tools-beyond-the-guide)

In [ ]:
def select_tool(task: dict) -> tuple[str, dict]:
    """Return (tool_name, tool_input) for one task dict (see the table above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        def call(task):
            out = select_tool(task)
            assert isinstance(out, tuple) and len(out) == 2 and isinstance(out[1], dict), \
                f"select_tool must return (tool_name, input_dict); got {out!r}"
            return out

        t, i = call({"kind": "find_files", "pattern": "**/*.test.tsx"})
        assert (t, i) == ("Glob", {"pattern": "**/*.test.tsx"}), f"file-name patterns are a Glob job with input {{'pattern': ...}}; got {t} {i}"
        t, i = call({"kind": "find_files", "pattern": "*.yaml", "under": "deploy"})
        assert t == "Glob" and i.get("pattern") == "*.yaml" and i.get("path") == "deploy", "Glob with the `pattern`, and the directory as Glob's `path`"
        t, i = call({"kind": "find_text", "text": "total(amount)", "file_type": "py"})
        assert t == "Grep", "searching for text inside files is a Grep job"
        assert i.get("output_mode") == "content" and i.get("type") == "py", "find_text wants matching lines (output_mode 'content') and the file type"
        assert isinstance(i.get("pattern"), str), "Grep needs `pattern` (the regex to search for)"
        rx = re.compile(i["pattern"])
        assert rx.search("x = total(amount)") and not rx.search("totalamount"), \
            "Grep patterns are regular expressions: escape the literal text (re.escape) so '(' does not become a group"
        t, i = call({"kind": "files_containing", "text": "legacy_auth"})
        assert t == "Grep" and i.get("output_mode", "files_with_matches") == "files_with_matches", "paths only: output_mode 'files_with_matches'"
        assert isinstance(i.get("pattern"), str) and re.search(i["pattern"], "import legacy_auth") \
            and not re.search(i["pattern"], "import legacy-auth"), "files_containing: Grep needs `pattern` = the escaped text"
        t, i = call({"kind": "count_matches", "text": "TODO", "under": "src"})
        assert t == "Grep" and i.get("output_mode") == "count" and i.get("path") == "src", "counts per file: output_mode 'count' plus `path`"
        assert isinstance(i.get("pattern"), str) and re.search(i["pattern"], "# TODO: fix") \
            and not re.search(i["pattern"], "# TO DO"), "count_matches: Grep needs `pattern` = the escaped text"
        assert call({"kind": "read_file", "path": "app/web.py"}) == ("Read", {"file_path": "app/web.py"}), "Read takes `file_path`"
        t, i = call({"kind": "create_file", "path": "app/new.py", "content": "X = 1\n"})
        assert (t, i) == ("Write", {"file_path": "app/new.py", "content": "X = 1\n"}), "new files: Write with file_path and content"
        t, i = call({"kind": "replace_once", "path": "a.py", "old": "x = 1", "new": "x = 2"})
        assert t == "Edit" and i.get("file_path") == "a.py" and i.get("old_string") == "x = 1" and i.get("new_string") == "x = 2" \
            and not i.get("replace_all"), "a targeted change is Edit with file_path, old_string, new_string (no replace_all)"
        t, i = call({"kind": "replace_every", "path": "b.py", "old": "y = 1", "new": "y = 2"})
        assert t == "Edit" and i.get("file_path") == "b.py" and i.get("old_string") == "y = 1" and i.get("new_string") == "y = 2" \
            and i.get("replace_all") is True, "changing every occurrence is Edit with file_path, old_string, new_string and replace_all: true"
        assert call({"kind": "run", "command": "pytest -q"}) == ("Bash", {"command": "pytest -q"}), "running commands is Bash"
        t, i = call({"kind": "edit_notebook_cell", "path": "nb.ipynb", "cell_id": "c7", "source": "print(2)"})
        assert t == "NotebookEdit" and i.get("notebook_path") == "nb.ipynb" and i.get("cell_id") == "c7" \
            and i.get("new_source") == "print(2)", "notebooks: NotebookEdit with notebook_path, cell_id, new_source (never Edit on .ipynb JSON)"
        t, i = call({"kind": "fetch_page", "url": "https://docs.python.org/3/", "question": "What changed in re?"})
        assert t == "WebFetch" and i.get("url") == "https://docs.python.org/3/" and isinstance(i.get("prompt"), str) \
            and "What changed in re?" in i["prompt"], "WebFetch needs the page `url` and a `prompt` that carries the question"
        assert call({"kind": "web_search", "query": "flask 3 migration"}) == ("WebSearch", {"query": "flask 3 migration"})
        question = "Find every external API call site and summarize them"
        t, i = call({"kind": "delegate_exploration", "question": question})
        assert t == "Agent" and i.get("subagent_type") == "Explore" and isinstance(i.get("prompt"), str) and question in i["prompt"] \
            and isinstance(i.get("description"), str) and i["description"].strip(), \
            "verbose read-only discovery goes to the Agent tool with subagent_type 'Explore', a short description and a prompt carrying the question"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Edit with a safe Read + Write fallback

Write `safe_edit(repo, path, old, new, occurrence=None)`:

1. If the file has not been read yet (`path not in repo.read_at`), Read it.
2. Try an `Edit` (no `replace_all`). Success → return `"edited"`.
3. If the Edit failed because the text matched **more than once** (the error mentions `matches`):
   - `occurrence is None` → return `"ambiguous"` and change nothing;
   - otherwise do the **fallback**: a fresh full Read (`repo.read_raw`), replace only the `occurrence`-th match (**1-based**) in code, `Write` the whole file, then **verify** by reading it back; return `"rewrote"`. An occurrence out of range → `"not_found"`, no Write.
4. Any other failure (text not found) → `"not_found"`, no Write.

Theory: [13.3](README.md#133-read-write-and-edit-full-file-vs-targeted-changes), [13.4](README.md#134-when-edit-fails-anchors-replace_all-and-the-read--write-fallback)

In [ ]:
def safe_edit(repo: FakeRepo, path: str, old: str, new: str, occurrence: int | None = None) -> str:
    """Return 'edited', 'rewrote', 'ambiguous' or 'not_found' (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        conf, pol = "deploy/nginx.conf", "app/billing/policy.py"
        r = hw_repo()
        before = r.files[pol]
        assert safe_edit(r, pol, "MAX_REFUND = 500.0", "MAX_REFUND = 650.0") == "edited", "a unique match should simply be 'edited'"
        assert r.files[pol] == before.replace("500.0", "650.0", 1), "the unique edit was not applied correctly"
        assert r.ledger[0].tool == "Read", "read the file before the first Edit (read-before-edit)"
        assert not r.calls("Write"), "a unique match needs Edit only, never a full-file Write"

        r = hw_repo(); orig = r.files[conf]
        out = safe_edit(r, conf, "proxy_read_timeout 30s;", "proxy_read_timeout 60s;")
        assert out == "ambiguous", f"two matches and no occurrence given -> 'ambiguous'; got {out!r}"
        assert r.files[conf] == orig and not r.calls("Write"), "when the target is ambiguous, change nothing"

        r = hw_repo(); orig = r.files[conf]
        out = safe_edit(r, conf, "proxy_read_timeout 30s;", "proxy_read_timeout 60s;", occurrence=2)
        assert out == "rewrote", f"occurrence=2 should use the Read + Write fallback and return 'rewrote'; got {out!r}"
        first, second = orig.find("proxy_read_timeout 30s;"), orig.rfind("proxy_read_timeout 30s;")
        expected = orig[:second] + "proxy_read_timeout 60s;" + orig[second + len("proxy_read_timeout 30s;"):]
        assert r.files[conf] == expected, "only the 2nd occurrence (1-based) may change; everything else must be byte-identical"
        tools = [c.tool for c in r.ledger]
        w = tools.index("Write")
        assert "Read" in tools[tools.index("Edit") + 1:w], "do a fresh, full Read after the failed Edit and before the Write"
        assert "Read" in tools[w + 1:], "verify after writing: read the file back"
        assert not any(c.tool == "Edit" and c.input.get("replace_all") for c in r.ledger), "never use replace_all to change one of several occurrences"

        r = hw_repo(); orig = r.files[conf]
        assert safe_edit(r, conf, "proxy_read_timeout 99s;", "x") == "not_found", "text that is not in the file -> 'not_found'"
        assert safe_edit(r, conf, "proxy_read_timeout 30s;", "x", occurrence=5) == "not_found", "occurrence out of range -> 'not_found'"
        assert r.files[conf] == orig and not r.calls("Write"), "nothing may be written when the target is not found"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Trace callers through a wrapper module

`app/pricing/__init__.py` (see `HW_EXTRA_FILES`) re-exports functions under new names, uses a **parenthesized multi-line import**, and lists one extra name in `__all__`.

- `exported_names(init_source)` → every exported name, each once: the name after `as` when there is an alias, the plain name otherwise, plus every `__all__` entry.
- `trace_wrapper(repo, init_path)` → `{exported_name: sorted list of .py files that mention it}`, found with **one Grep per name** using word boundaries (`\bname\b`, `type="py"`), excluding files inside the wrapper's own package directory. Read only the wrapper itself.

Theory: [13.7](README.md#137-tracing-through-wrapper-modules)

In [ ]:
def exported_names(init_source: str) -> list[str]:
    """Every name the wrapper exports: import aliases (also inside parentheses) and __all__ entries, no duplicates."""
    # TODO
    raise NotImplementedError


def trace_wrapper(repo: FakeRepo, init_path: str) -> dict[str, list[str]]:
    """Map each exported name -> sorted .py files OUTSIDE the package that mention it (found with Grep)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        names = exported_names(HW_EXTRA_FILES["app/pricing/__init__.py"])
        assert isinstance(names, list), "exported_names must return a list"
        expected = ["quote", "apply_discount", "tax_rate", "PRICE_VERSION", "round_money"]
        assert "quote" in names and "compute_price" not in names, "an alias is exported under its NEW name: 'compute_price as quote' exports 'quote'"
        assert "apply_discount" in names, "names inside a parenthesized multi-line import are exported too"
        assert "PRICE_VERSION" in names, "read __all__ as well: it lists names that are not imported (PRICE_VERSION)"
        assert sorted(names) == sorted(expected) and len(names) == len(set(names)), f"expected each of {expected} exactly once; got {names}"
        r = hw_repo()
        result = trace_wrapper(r, "app/pricing/__init__.py")
        want = {"quote": ["app/api/pricing_routes.py", "app/reports/monthly.py"],
                "apply_discount": ["app/checkout/cart.py", "tests/test_pricing.py"],
                "tax_rate": ["app/api/pricing_routes.py"], "round_money": ["app/checkout/cart.py"],
                "PRICE_VERSION": ["app/api/pricing_routes.py"]}
        for name, files in want.items():
            assert result.get(name) == files, f"{name}: expected {files}, got {result.get(name)} (search with \\b word boundaries, sort, exclude app/pricing/)"
        greps = r.calls("Grep")
        assert len(greps) >= 5, "search for each exported name with the Grep tool (repo.call('Grep', ...))"
        assert {c.input["file_path"] for c in r.calls("Read")} <= {"app/pricing/__init__.py"}, \
            "read only the wrapper itself; the callers come from Grep results, not from reading files"
    _run_check(3, check)


check_ex3()

### Exercise 4 — An incremental investigation with a read budget

Write `investigate(repo, error_text)` returning `{"raised_in": "path:line", "function": name, "callers": [...]}`:

1. Your **first** call is a Grep for the error text (escape it: some messages contain regex metacharacters) in `content` mode.
2. Read the file it surfaced and find the enclosing function (the nearest `def name(` above the line).
3. Grep for calls `\bname\(` in Python files. `callers` = sorted files with such a call, excluding the defining file, anything under `tests/` or `legacy/`, and `def` lines.

Constraints the checker enforces: no Glob, **at most 2 Reads**, and only Read files that an earlier Grep result named.

Theory: [13.6](README.md#136-incremental-investigation-grep-read-repeat)

In [ ]:
def investigate(repo: FakeRepo, error_text: str) -> dict:
    """Return {"raised_in": "path:line", "function": name, "callers": [sorted paths]} (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        cases = [
            ("Refund declined by processor", {"raised_in": "app/payments/gateway.py:18", "function": "submit_refund",
                                              "callers": ["app/billing/refunds.py"]}),
            ("Ledger entry rejected: account closed", {"raised_in": "app/billing/ledger.py:6", "function": "post_entry",
                                                       "callers": ["app/billing/refunds.py", "app/workers/nightly.py"]}),
            ("Coupon expired (code reuse)", {"raised_in": "app/pricing/engine.py:14", "function": "apply_discount",
                                             "callers": ["app/checkout/cart.py"]}),
        ]
        for text, want in cases:
            r = hw_repo()
            try:
                got = investigate(r, text)
            except NotImplementedError:
                raise
            except Exception as exc:  # noqa: BLE001
                raise AssertionError(f"{text!r}: your code raised {type(exc).__name__}: {exc}"
                                     + (" (did you escape the error text? it contains '(')" if "(" in text else "")) from None
            assert isinstance(got, dict), "investigate must return a dict"
            for k in ("raised_in", "function", "callers"):
                assert got.get(k) == want[k], f"{text!r}: {k} expected {want[k]!r}, got {got.get(k)!r}" + \
                    (" (escape the text: it contains regex metacharacters)" if "(" in text and k == "raised_in" else "") + \
                    (" (exclude the defining file, tests/, legacy/ and def lines)" if k == "callers" else "")
            assert r.ledger and r.ledger[0].tool == "Grep", "start with a search (Grep), not with reading files"
            assert not r.calls("Glob"), "no Glob needed: the error text is a content search"
            reads = r.calls("Read")
            assert len(reads) <= 2, f"read budget is 2 files; you read {len(reads)}"
            for c in reads:
                seen = " ".join(x.output for x in r.ledger[: r.ledger.index(c)] if x.tool == "Grep")
                assert c.input["file_path"] in seen, f"only Read files that an earlier Grep surfaced ({c.input['file_path']})"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Permission rules for a modernization team

Write `build_permissions()` returning `{"allow": [...], "ask": [...], "deny": [...]}` (Claude Code rule syntax) so that:

1. Claude never reads `.env` at the project root, or anything under a `secrets/` directory at **any** depth (and therefore cannot change them with Edit or Write either).
2. Claude never changes files under `migrations/` at the project root, with **any** file tool (Edit, Write, NotebookEdit).
3. `npm run test` (with or without extra arguments) and `npm run lint` run without a prompt; other `npm` commands still prompt.
4. `git push` always asks.
5. `curl` never runs, including inside compound commands.
6. Pages on `docs.python.org` are fetched without a prompt; other domains still prompt.
7. Ordinary edits elsewhere still prompt: do not deny or allow `Edit` as a whole.

The checker uses the notebook's simplified evaluator (`decide`): deny → ask → allow; `/path` anchors at the project root; a bare directory pattern such as `dir/**` in a deny rule matches at any depth. Theory: [13.10](README.md#1310-permission-rules-per-tool-beyond-the-guide)

In [ ]:
def build_permissions() -> dict[str, list[str]]:
    """Return {"allow": [...], "ask": [...], "deny": [...]} meeting the seven requirements above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        rules = build_permissions()
        assert isinstance(rules, dict) and set(rules) <= {"allow", "ask", "deny"}, "return a dict with allow / ask / deny lists"
        warnings = lint_rules(rules)
        assert not any("never consulted" in w for w in warnings), f"{warnings[0]} (only Read(...) and Edit(...) are path rules)"
        cases = [
            ("Read", {"file_path": ".env"}, "deny", "requirement 1: reading .env at the root"),
            ("Edit", {"file_path": ".env"}, "deny", "requirement 1: a Read deny also blocks edits of .env"),
            ("Grep", {"pattern": "KEY", "path": "config/secrets"}, "deny", "requirement 1: secrets/ at ANY depth (Read rules cover Grep)"),
            ("Read", {"file_path": "app/secrets/token.txt"}, "deny", "requirement 1: secrets/ at any depth"),
            ("Edit", {"file_path": "app/billing/policy.py"}, "ask", "requirement 7: ordinary edits must still prompt, not be denied or auto-allowed"),
            ("Write", {"file_path": "migrations/0042_add_index.sql"}, "deny", "requirement 2: Write under migrations/ (use an Edit(...) rule)"),
            ("NotebookEdit", {"notebook_path": "migrations/check.ipynb"}, "deny", "requirement 2: every file tool"),
            ("Read", {"file_path": "app/billing/policy.py"}, "allow", "normal reads must keep working"),
            ("Bash", {"command": "npm run test"}, "allow", "requirement 3: npm run test"),
            ("Bash", {"command": "npm run test -- --coverage"}, "allow", "requirement 3: npm run test with arguments"),
            ("Bash", {"command": "npm run lint"}, "allow", "requirement 3: npm run lint"),
            ("Bash", {"command": "npm install left-pad"}, "ask", "requirement 3: other npm commands must still prompt (Bash(npm *) is too broad)"),
            ("Bash", {"command": "git push origin main"}, "ask", "requirement 4: ask before git push"),
            ("Bash", {"command": "curl https://example.com"}, "deny", "requirement 5: never curl"),
            ("Bash", {"command": "npm run lint && curl https://example.com"}, "deny", "requirement 5: also inside compound commands"),
            ("WebFetch", {"url": "https://docs.python.org/3/library/re.html"}, "allow", "requirement 6: docs.python.org"),
            ("WebFetch", {"url": "https://example.org/"}, "ask", "requirement 6: other domains must still prompt (no bare WebFetch allow)"),
        ]
        for tool, inp, want, why in cases:
            got = decide(rules, tool, inp)
            assert got == want, f"{why}: {tool}({next(iter(inp.values()))}) -> {got}, expected {want}"
    _run_check(5, check)


check_ex5()

### Exercise 6 — Rank modernization hotspots from tool evidence

Write `rank_hotspots(repo, modules, churn)` returning one dict per module, `{"module", "fan_in", "churn", "has_tests", "score"}`:

- `fan_in` = `fan_in(repo, module)` (helper provided);
- `churn` = `churn.get(module, 0)`;
- `has_tests` = whether **Glob** `tests/**/test_<stem>.py` finds a file (`<stem>` = file name without `.py`);
- `score = (fan_in + 1) * (churn + 1) * (2 if no tests else 1) * (2 if the module is under app/billing/ or app/payments/ else 1)`;
- sorted by `score` descending, then `module` ascending.

Theory: [13.11](README.md#1311-codebase-understanding-for-modernization-work)

In [ ]:
def rank_hotspots(repo: FakeRepo, modules: list[str], churn: dict[str, int]) -> list[dict]:
    """Return one dict per module, sorted by score (desc), then module (asc). Rules above."""
    # TODO
    raise NotImplementedError

In [ ]:
EXPECTED_ORDER = ['app/billing/refunds.py', 'app/payments/gateway.py', 'app/billing/ledger.py', 'app/api/routes.py', 'app/orders/service.py', 'app/pricing/engine.py']  # used by the checker

def check_ex6() -> None:
    FIELDS = ("module", "fan_in", "churn", "has_tests", "score")

    def expected_rows(modules, churn):
        # Recomputed from the repository itself (not from your function), so every field can be checked.
        ref = hw_repo()
        rows = []
        for m in modules:
            stem = Path(m).stem
            has_tests = any(re.fullmatch(rf"tests/(?:.*/)?test_{re.escape(stem)}\.py", f) for f in ref.files)
            fi, ch = fan_in(ref, m), churn.get(m, 0)
            risk = 2 if m.startswith(("app/billing/", "app/payments/")) else 1
            rows.append({"module": m, "fan_in": fi, "churn": ch, "has_tests": has_tests,
                         "score": (fi + 1) * (ch + 1) * (1 if has_tests else 2) * risk})
        return sorted(rows, key=lambda x: (-x["score"], x["module"]))

    def compare(rows, modules, churn, label):
        assert isinstance(rows, list) and len(rows) == len(modules), f"{label}: one dict per module"
        assert all(isinstance(x, dict) and set(FIELDS) <= set(x) for x in rows), \
            f"{label}: each row needs module, fan_in, churn, has_tests, score"
        got = [x["module"] for x in rows]
        assert sorted(got) == sorted(modules), f"{label}: every module exactly once (got {got})"
        want = expected_rows(modules, churn)
        by = {x["module"]: x for x in want}
        for x in rows:
            for k in FIELDS[1:]:
                assert x[k] == by[x["module"]][k], \
                    f"{label}: {x['module']} has {k} = {x[k]!r}, expected {by[x['module']][k]!r} (rules in the task above)"
        order = [x["module"] for x in want]
        assert got == order, f"{label}: sort by score (desc), then module (asc); expected {order}, got {got}"

    def check():
        r = hw_repo()
        modules = ["app/billing/refunds.py", "app/payments/gateway.py", "app/billing/ledger.py",
                   "app/orders/service.py", "app/pricing/engine.py", "app/api/routes.py"]
        rows = rank_hotspots(r, modules, ACME_GIT_LOG)
        assert isinstance(rows, list) and len(rows) == len(modules), "one dict per module"
        assert all(isinstance(x, dict) and set(FIELDS) <= set(x) for x in rows), \
            "each row needs module, fan_in, churn, has_tests, score"
        by = {x["module"]: x for x in rows}
        assert set(by) == set(modules), "every module exactly once"
        assert by["app/payments/gateway.py"]["has_tests"] is False and by["app/billing/refunds.py"]["has_tests"] is True, \
            "has_tests: Glob tests/**/test_<stem>.py"
        assert len(r.calls("Glob")) >= len(modules), "use the Glob tool to look for each module's test file"
        assert by["app/pricing/engine.py"]["churn"] == 0, "a module missing from the churn dict has churn 0"
        assert by["app/payments/gateway.py"]["score"] == 192, \
            f"gateway: (fan_in 1 + 1) * (churn 23 + 1) * 2 (no tests) * 2 (money) = 192; got {by['app/payments/gateway.py']['score']}"
        compare(rows, modules, ACME_GIT_LOG, "git-log churn")
        assert [x["module"] for x in rows] == EXPECTED_ORDER, f"expected order {EXPECTED_ORDER}"
        # Tie case: no churn data and the modules in reverse order, so equal scores must be ordered by module name.
        tie_modules = list(reversed(modules))
        compare(rank_hotspots(hw_repo(), tie_modules, {}), tie_modules, {}, "tie case (empty churn)")
    _run_check(6, check)


check_ex6()

## Part C — Architecture scenario

**Mapping a legacy claims platform.** An insurer runs a 14-year-old claims platform: about 6,000 files of Java, Perl batch scripts and SQL stored procedures, little documentation, and none of the original authors left. The business wants the *claim payout* capability extracted into a separate service within six months. You get a read-only mirror of the repository, Claude Code for the engineering team, and budget for an Agent SDK job that can run overnight.

Describe your approach (15–25 lines):

1. How the team and Claude build an accurate map of the payout flow: which tools in which order, starting from what.
2. How you handle indirection: dependency-injection wiring, re-exported facades, stored procedures called by name, Perl scripts invoked from cron.
3. How you keep the investigation within context limits over several weeks and many sessions.
4. How you configure tools and permissions for the exploration phase and for the later change phase.
5. What the deliverable of the mapping phase is, and how you know the map is complete enough to plan the extraction.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Start from concrete anchors, search before reading**: payout endpoints and screens, the payout tables' names, error and audit messages, batch job names in crontab files; Glob for the structure (`**/*.java`, `**/*.pl`, `**/*.sql`, build files), Grep for entry points and table names (`files_with_matches` first, then `content`), Read only the hits, follow imports and calls with further searches.
- **Indirection**: resolve DI configuration (XML or annotations) to concrete classes; list every name a facade or wrapper exports and search for each; search SQL and code for stored-procedure names as strings; Grep crontab and shell wrappers to connect Perl scripts to schedules; note dynamic dispatch that static search cannot resolve and confirm it with runtime evidence (logs) or people; consider a code intelligence (LSP) plugin for Java references.
- **Context management**: delegate discovery to the Explore subagent or read-only custom subagents (one per question: "all writers of the PAYOUT table"); a shared scratchpad or findings file with file:line evidence, read back at each session start; phase summaries before the next phase; `/compact` only as a fallback; Read deny rules for generated and vendored code; structured state so an overnight Agent SDK job can resume.
- **Tools and permissions**: exploration agents with `tools` limited to Read, Grep, Glob (plus read-only Bash such as `git log`), `dontAsk` for unattended runs, `max_turns` / `max_budget_usd`; no Write/Edit on the mirror; deny reads of credentials (`Read(**/*.properties)` with secrets, `.env`); later, a change phase on a branch with Edit allowed, `Edit` deny rules for migrations and generated code, `ask` for `git push`, sandbox or hooks for real enforcement, never `bypassPermissions` outside an isolated container.
- **Deliverable**: a dependency map of the payout flow (entry points → services → data stores → external systems), the list of shared tables and callers that block extraction, hotspots ranked by fan-in, churn and test gaps, unknowns with owners, and a prioritized plan written in plan mode that adapts as dependencies appear.
- **Completeness checks**: every writer and reader of the payout tables accounted for (Grep counts reconcile), every exported name of each facade searched, cron and stored-procedure callers traced, spot checks against production logs, and review with the client's engineers.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to exam prep" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next: [exam prep](../exam-prep/README.md) (certification ladder, program rules and per-exam study folders). Back to the [Claude path overview](../README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** "Which files import X" is a content question; scoping Grep by the `*.spec.ts` glob and asking for paths only returns exactly the files to change. A reads dozens of files that do not use the helper. C matches the word "spec" anywhere. D assumes an index file that lists every test.

**Q2:** **D.** Glob results are capped at 100 and the result says so; a capped list is not evidence of absence. Narrow the pattern or the `path`. A ignores the truncation notice. B fails because Read does not list directories (use `ls` or Glob). C searches file contents for ".ts", which is not the question.

**Q3:** **C.** The repeated text cannot be anchored with a small unique context, so the reliable path is Read the full file, change the right occurrence by structure, Write the whole file, and verify. A changes all 14 stubs. B repeats a deterministic failure (or matches nothing). D edits a line number that may no longer point at the same code.

**Q4:** **A.** The Read output's line numbers and the tab after them are display only, not file content. `old_string` must be the file's exact text, indentation included. B and C do not address the cause (and C risks losing content). D finds files by path, not lines.

**Q5:** **B.** Reading everything floods the context and degrades answers toward generic patterns (task 5.4); incremental search from entry points plus a scratchpad keeps the context on evidence. A and D treat the symptom. C loses the specific details the question needs.

**Q6:** **C.** The barrel re-exports `chargeCard` as `charge`, so consumers use the alias; list every exported name for the function and search for each (task 2.5). A acts on an incomplete search. B also matches `discharge`, `chargeback` and so on. D reads far more than needed and still requires you to know which name to look for.

**Q7:** **D.** Agents prefer familiar built-in tools when an MCP tool's description gives no reason to choose it; the description must state the unique, current data and when to prefer the tool (task 2.4). A (a prompt instruction) and B (a deny rule) both stop Grep from being used, even though Grep is still right for code searches; and A enforces nothing. C adds machinery for a description problem.

**Q8:** **A.** A `Read(...)` deny rule blocks Claude's file reads of that path and also blocks Edit and Write on it, including creating the file (NotebookEdit is not covered; add an `Edit(...)` deny when no tool at all may change a path). `/.env` anchors at the project root for project settings. B is a `Write(path)` rule, which file permission checks never consult. C is not how negation works (and an allow cannot carve out a deny). D shapes behavior but enforces nothing.

**Q9:** **B.** A line-number edit is blind: if the file changed, it silently changes whatever is on line 212 now. Edit matches the exact text and fails safely when it is gone. A is false. C replaces the whole file, which is a bigger blast radius. D is a search tool, not an editing tool.

**Q10:** **C.** Modernization starts with evidence (where the framework is used, what changes often, what is untested), then a prioritized plan in plan mode, then tested slices that adapt as dependencies appear. A and B ignore risk and dependencies. D cannot fit in a context window and degrades answer quality.
</details>

<details><summary>Exercise 1 — The right tool with the right input: reference solution</summary>

```python
def select_tool(task: dict) -> tuple[str, dict]:
    kind = task["kind"]
    if kind == "find_files":
        inp = {"pattern": task["pattern"]}
        if task.get("under"):
            inp["path"] = task["under"]
        return "Glob", inp
    if kind == "find_text":
        inp = {"pattern": re.escape(task["text"]), "output_mode": "content"}
        if task.get("file_type"):
            inp["type"] = task["file_type"]
        return "Grep", inp
    if kind == "files_containing":
        return "Grep", {"pattern": re.escape(task["text"]), "output_mode": "files_with_matches"}
    if kind == "count_matches":
        return "Grep", {"pattern": re.escape(task["text"]), "path": task["under"], "output_mode": "count"}
    if kind == "read_file":
        return "Read", {"file_path": task["path"]}
    if kind == "create_file":
        return "Write", {"file_path": task["path"], "content": task["content"]}
    if kind in ("replace_once", "replace_every"):
        inp = {"file_path": task["path"], "old_string": task["old"], "new_string": task["new"]}
        if kind == "replace_every":
            inp["replace_all"] = True
        return "Edit", inp
    if kind == "run":
        return "Bash", {"command": task["command"]}
    if kind == "edit_notebook_cell":
        return "NotebookEdit", {"notebook_path": task["path"], "cell_id": task["cell_id"],
                                "new_source": task["source"], "edit_mode": "replace"}
    if kind == "fetch_page":
        return "WebFetch", {"url": task["url"], "prompt": task["question"]}
    if kind == "web_search":
        return "WebSearch", {"query": task["query"]}
    if kind == "delegate_exploration":
        return "Agent", {"description": task["question"][:40], "prompt": task["question"], "subagent_type": "Explore"}
    raise ValueError(f"unknown task kind {kind}")
```
</details>

<details><summary>Exercise 2 — Edit with a safe Read + Write fallback: reference solution</summary>

```python
def safe_edit(repo: FakeRepo, path: str, old: str, new: str, occurrence: int | None = None) -> str:
    if path not in repo.read_at:
        repo.call("Read", file_path=path)
    res = repo.call("Edit", file_path=path, old_string=old, new_string=new)
    if res.ok:
        return "edited"
    if "matches" not in res.output:
        return "not_found"
    if occurrence is None:
        return "ambiguous"
    text = repo.read_raw(path)                       # fresh, full Read right before the Write
    if text is None:
        return "not_found"
    starts, i = [], text.find(old)
    while i != -1:
        starts.append(i)
        i = text.find(old, i + 1)
    if not 1 <= occurrence <= len(starts):
        return "not_found"
    at = starts[occurrence - 1]
    updated = text[:at] + new + text[at + len(old):]
    if not repo.call("Write", file_path=path, content=updated).ok:
        return "not_found"
    return "rewrote" if repo.read_raw(path) == updated else "not_found"
```
</details>

<details><summary>Exercise 3 — Trace callers through a wrapper module: reference solution</summary>

```python
def exported_names(init_source: str) -> list[str]:
    names = []
    flat = re.sub(r"\(\s*([^)]*?)\s*\)", lambda m: m.group(1).replace("\n", " "), init_source)
    for group in re.findall(r"^from\s+\S+\s+import\s+(.+)$", flat, re.M):
        for item in group.split(","):
            item = item.split("#")[0].strip()
            if item:
                names.append(item.split(" as ")[-1].strip())
    m = re.search(r"__all__\s*=\s*\[([^\]]*)\]", init_source)
    if m:
        names += re.findall(r"[\"'](\w+)[\"']", m.group(1))
    return list(dict.fromkeys(names))


def trace_wrapper(repo: FakeRepo, init_path: str) -> dict[str, list[str]]:
    package_dir = init_path.rsplit("/", 1)[0] + "/"
    source = repo.read_raw(init_path)
    result = {}
    for name in exported_names(source):
        res = repo.call("Grep", pattern=rf"\b{re.escape(name)}\b", type="py")
        result[name] = sorted(f for f in res.output.splitlines() if f.endswith(".py") and not f.startswith(package_dir))
    return result
```
</details>

<details><summary>Exercise 4 — An incremental investigation with a read budget: reference solution</summary>

```python
def investigate(repo: FakeRepo, error_text: str) -> dict:
    hit = repo.call("Grep", pattern=re.escape(error_text), output_mode="content").output.splitlines()[0]
    path, line, _ = hit.split(":", 2)
    lines = repo.read_raw(path).splitlines()
    function = None
    for i in range(int(line) - 1, -1, -1):
        m = re.match(r"\s*def (\w+)\(", lines[i])
        if m:
            function = m.group(1)
            break
    calls = repo.call("Grep", pattern=rf"\b{function}\(", type="py", output_mode="content").output.splitlines()
    callers = set()
    for row in calls:
        m = re.match(r"([^:]+):(\d+):(.*)", row)
        if m and m.group(1) != path and not m.group(1).startswith(("tests/", "legacy/")) \
                and not re.match(r"\s*def ", m.group(3)):
            callers.add(m.group(1))
    return {"raised_in": f"{path}:{line}", "function": function, "callers": sorted(callers)}
```
</details>

<details><summary>Exercise 5 — Permission rules for a modernization team: reference solution</summary>

```python
def build_permissions() -> dict[str, list[str]]:
    return {
        "allow": ["Bash(npm run test *)", "Bash(npm run lint)", "WebFetch(domain:docs.python.org)"],
        "ask": ["Bash(git push *)"],
        "deny": ["Read(/.env)", "Read(secrets/**)", "Edit(/migrations/**)", "Bash(curl *)"],
    }
```
</details>

<details><summary>Exercise 6 — Rank modernization hotspots from tool evidence: reference solution</summary>

```python
def rank_hotspots(repo: FakeRepo, modules: list[str], churn: dict[str, int]) -> list[dict]:
    rows = []
    for m in modules:
        stem = Path(m).stem
        has_tests = repo.call("Glob", pattern=f"tests/**/test_{stem}.py").output != "No files found"
        fi, ch = fan_in(repo, m), churn.get(m, 0)
        risk = 2 if m.startswith(("app/billing/", "app/payments/")) else 1
        score = (fi + 1) * (ch + 1) * (1 if has_tests else 2) * risk
        rows.append({"module": m, "fan_in": fi, "churn": ch, "has_tests": has_tests, "score": score})
    return sorted(rows, key=lambda r: (-r["score"], r["module"]))
```
</details>